# DoctrineRAG: Qwen3-8B + 3군 LoRA Adapter + vLLM + ngrok

Google Drive에 저장된 3개 LoRA Adapter를 Qwen3-8B에 붙여 Colab에서 OpenAI-compatible API로 서빙합니다.

```txt
vLLM server : port 8000
model="army" -> Qwen3-8B + army LoRA
model="navy" -> Qwen3-8B + navy LoRA
model="air"  -> Qwen3-8B + air LoRA
```

로컬 백엔드는 ngrok URL의 `/v1/chat/completions`로 요청하고, `model` 값으로 군별 adapter를 선택합니다.


## 0. GPU 확인

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM GB: 79.25


## 1. 패키지 설치

In [2]:
!pip install -q -U vllm pyngrok openai requests huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.2/248.2 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 116.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 748.7/748

## 2. Google Drive 마운트

In [3]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


## 3. Base Model 및 LoRA 경로 설정

Drive 경로가 다르면 아래 3개 경로를 수정하세요.

각 LoRA 폴더에는 최소 아래 파일이 있어야 합니다.

```txt
adapter_config.json
adapter_model.safetensors
```


In [4]:
from pathlib import Path

BASE_MODEL = "Qwen/Qwen3-8B"

ARMY_LORA = "/content/drive/MyDrive/doctor-cache/lora_adapters_qwen3_8b/air_lora_adapter_qwen3_8b"
NAVY_LORA = "/content/drive/MyDrive/doctor-cache/lora_adapters_qwen3_8b/navy_lora_adapter_qwen3_8b"
AIR_LORA = "/content/drive/MyDrive/doctor-cache/lora_adapters_qwen3_8b/air_lora_adapter_qwen3_8b"

for name, path in {
    "army": ARMY_LORA,
    "navy": NAVY_LORA,
    "air": AIR_LORA,
}.items():
    p = Path(path)
    print(f"[{name}] {path}")
    print("  exists:", p.exists())
    print("  adapter_config.json:", (p / "adapter_config.json").exists())
    print("  adapter_model.safetensors:", (p / "adapter_model.safetensors").exists())
    print()


[army] /content/drive/MyDrive/doctor-cache/lora_adapters_qwen3_8b/air_lora_adapter_qwen3_8b
  exists: True
  adapter_config.json: True
  adapter_model.safetensors: True

[navy] /content/drive/MyDrive/doctor-cache/lora_adapters_qwen3_8b/navy_lora_adapter_qwen3_8b
  exists: True
  adapter_config.json: True
  adapter_model.safetensors: True

[air] /content/drive/MyDrive/doctor-cache/lora_adapters_qwen3_8b/air_lora_adapter_qwen3_8b
  exists: True
  adapter_config.json: True
  adapter_model.safetensors: True



## 4. HuggingFace 로그인

Qwen3-8B 다운로드를 위해 HuggingFace token 로그인을 권장합니다. 입력창에는 비밀번호가 아니라 `hf_...` access token을 넣으세요.

Public 모델 접근만 필요하면 이 셀을 건너뛰어도 됩니다.


In [5]:
from huggingface_hub import login

login()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## 5. vLLM 서버 실행

하나의 서버에 3개 LoRA를 모두 등록합니다.

등록명:
- `army`
- `navy`
- `air`

이 이름이 API 요청의 `model` 값입니다.


In [5]:
import subprocess, time

VLLM_PORT = "8000"

process = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", BASE_MODEL,
        "--host", "0.0.0.0",
        "--port", VLLM_PORT,
        "--enable-lora",
        "--lora-modules",
        f"army={ARMY_LORA}",
        f"navy={NAVY_LORA}",
        f"air={AIR_LORA}",
        "--max-model-len", "4096",
        "--gpu-memory-utilization", "0.85",
        "--trust-remote-code",
    ],
    stdout=open("/content/vllm_stdout.log", "w"),
    stderr=open("/content/vllm_stderr.log", "w"),
)

time.sleep(20)
print("poll:", process.poll())

poll: None


In [6]:
!lsof -i:8000
!ps -ef | grep vllm

root       10789    9471 99 04:06 ?        00:00:19 python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-8B --host 0.0.0.0 --port 8000 --enable-lora --lora-modules army=/content/drive/MyDrive/doctor-cache/lora_adapters_qwen3_8b/air_lora_adapter_qwen3_8b navy=/content/drive/MyDrive/doctor-cache/lora_adapters_qwen3_8b/navy_lora_adapter_qwen3_8b air=/content/drive/MyDrive/doctor-cache/lora_adapters_qwen3_8b/air_lora_adapter_qwen3_8b --max-model-len 4096 --gpu-memory-utilization 0.85 --trust-remote-code
root       10932   10789 99 04:06 ?        00:00:03 /usr/bin/python3 -m vllm.model_executor.models.registry
root       10957    9471  0 04:06 ?        00:00:00 /bin/bash -c ps -ef | grep vllm
root       10959   10957  0 04:06 ?        00:00:00 grep vllm


In [7]:
print(process.poll())

None


In [73]:
!tail -100 /content/vllm_stderr.log

(APIServer pid=10789) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(EngineCore pid=11132) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:01<00:04,  1.10s/it]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:02<00:03,  1.15s/it]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:03<00:02,  1.17s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:04<00:01,  1.09s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:04<00:00,  1.19it/s]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:04<00:00,  1.03it/s]
(EngineCore pid=11132) 
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):  66%|██████

## 6. vLLM 로그 확인

In [26]:
import time

start = time.time()
while time.time() - start < 30:
    line = vllm_process.stdout.readline()
    if not line:
        break
    print(line.rstrip())
print("--- log check done ---")


NameError: name 'vllm_process' is not defined

In [71]:
import requests

r = requests.get("http://localhost:8000/v1/models", timeout=10)
print(r.status_code)
print(r.text)

ConnectionError: HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /v1/models (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7a602d34cad0>: Failed to establish a new connection: [Errno 111] Connection refused'))

## 7. 로컬 API 상태 확인

In [10]:
import requests
import json
import time

LOCAL_BASE_URL = "http://localhost:8000/v1"

for i in range(40):
    try:
        r = requests.get(f"{LOCAL_BASE_URL}/models", timeout=5)
        print("status:", r.status_code)
        print(json.dumps(r.json(), indent=2, ensure_ascii=False))
        break
    except Exception as e:
        print(f"waiting... {i+1}/40", repr(e))
        time.sleep(5)
else:
    raise RuntimeError("vLLM 서버가 응답하지 않습니다. 로그를 확인하세요.")


waiting... 1/40 ConnectionError(MaxRetryError("HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /v1/models (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7a616c5fda90>: Failed to establish a new connection: [Errno 111] Connection refused'))"))


KeyboardInterrupt: 

## 8. ngrok 연결

`YOUR_NGROK_TOKEN`을 본인 토큰으로 바꿔 실행하세요.


In [28]:
from pyngrok import ngrok

NGROK_TOKEN = "YOUR_NGROK_TOKEN"

ngrok.kill()
ngrok.set_auth_token("3DNFaHpYfJ5Cijzfw8YAOU9fSCX_2hdhzL4aWLffNi8iiHrQ4")

tunnel = ngrok.connect(8000, proto="http")
PUBLIC_BASE_URL = tunnel.public_url

print("Public URL:", PUBLIC_BASE_URL)
print("API Base URL:", f"{PUBLIC_BASE_URL}/v1")


Public URL: https://chomp-ethics-dazzling.ngrok-free.dev
API Base URL: https://chomp-ethics-dazzling.ngrok-free.dev/v1


## 9. Army/Navy/Air API 테스트

같은 API URL을 사용하고 `model` 값만 바꿉니다.


In [ ]:
from openai import OpenAI
import re

client = OpenAI(
    base_url=f"{PUBLIC_BASE_URL}/v1",
    api_key="EMPTY",
)

REQUIRED_SECTIONS = [
    "1. 개요",
    "2. 핵심 원칙",
    "3. 운용 절차",
    "4. 지휘관 고려사항",
    "5. 유의사항",
]


def clean_model_output(text: str) -> str:
    if not text:
        return ""

    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"</?think>", "", text)
    text = re.sub(r"<\|im_start\|>|<\|im_end\|>", "", text)

    return text.strip()


def remove_duplicate_sentences(text: str) -> str:
    """
    동일 문장이 여러 번 반복될 경우 첫 번째만 유지
    """
    sentences = re.split(r"(?<=[.!?。！？다])\s+", text)
    seen = set()
    cleaned = []

    for sentence in sentences:
        normalized = re.sub(r"\s+", " ", sentence).strip()

        if not normalized:
            continue

        if normalized not in seen:
            seen.add(normalized)
            cleaned.append(sentence.strip())

    return "\n".join(cleaned).strip()


def remove_duplicate_lines(text: str) -> str:
    """
    완전히 동일하거나 거의 동일한 줄 반복 제거
    """
    lines = text.splitlines()
    seen = set()
    cleaned = []

    for line in lines:
        normalized = re.sub(r"\s+", " ", line).strip()
        normalized = re.sub(r"[-*#`_]", "", normalized)

        if not normalized:
            cleaned.append(line)
            continue

        if normalized not in seen:
            seen.add(normalized)
            cleaned.append(line)

    return "\n".join(cleaned).strip()


def postprocess_answer(text: str) -> str:
    text = clean_model_output(text)
    text = remove_duplicate_lines(text)
    text = remove_duplicate_sentences(text)
    return text.strip()


def validate_answer(text: str) -> dict:
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    unique_lines = set(lines)

    repetition_ratio = 0
    if lines:
        repetition_ratio = 1 - (len(unique_lines) / len(lines))

    return {
        "has_think_tag": bool(re.search(r"</?think>", text)),
        "has_chat_template_token": bool(
            re.search(r"<\|im_start\||<\|im_end\|>", text)
        ),
        "has_required_sections": all(
            section in text for section in REQUIRED_SECTIONS
        ),
        "possible_fake_reference": bool(
            re.search(r"\bJP\s?\d+-\d+\b", text)
        ),
        "repetition_ratio": round(repetition_ratio, 2),
        "length": len(text),
    }


def build_system_prompt(role_prompt: str) -> str:
    return f"""
{role_prompt}

중요 지침:
- 내부 추론 과정은 절대 출력하지 마라.
- <think> 태그를 절대 출력하지 마라.
- 최종 답변만 출력하라.
- 같은 문장을 반복하지 마라.
- 같은 설명을 항목마다 복사해서 반복하지 마라.
- 각 항목은 서로 다른 내용으로 작성하라.
- 존재하지 않는 교리명, 문서번호, 출처를 임의로 만들지 마라.
- 확실하지 않은 내용은 "일반 원칙 기준"이라고 명시하라.
- 공개 가능한 일반 교리 수준에서만 설명하라.

답변 형식:
1. 개요
2. 핵심 원칙
3. 운용 절차
4. 지휘관 고려사항
5. 유의사항
""".strip()


def ask(
    model_name: str,
    system_prompt: str,
    question: str,
    max_tokens: int = 700,
    temperature: float = 0.4,
    return_raw: bool = False,
):
    guarded_system_prompt = build_system_prompt(system_prompt)

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": guarded_system_prompt},
            {"role": "user", "content": question.strip()},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
        frequency_penalty=0.8,
        presence_penalty=0.3,
    )

    raw = response.choices[0].message.content or ""
    answer = postprocess_answer(raw)
    validation = validate_answer(answer)

    if return_raw:
        return {
            "raw": raw,
            "answer": answer,
            "validation": validation,
        }

    return answer

In [ ]:
army_result = ask(
    "army",
    "너는 육군 교리 전문 AI 참모이다. 지상작전, 방어작전, 기동, 화력 운용 관점에서 답변하라.",
    "방어작전에서 예비대 운용 원칙을 설명하라.",
    max_tokens=700,
    return_raw=True,
)

print("=== CLEANED ANSWER ===")
print(army_result["answer"])

print("\n=== VALIDATION ===")
print(army_result["validation"])

=== CLEANED ANSWER ===
1.
개요  
방어작전에서 예비대는 지상군의 전투기능을 유지하고 방어선을 위협하는 적의 주력부대를 제거하기 위해 운용되는 부대이다.
2.
핵심 원칙  
- 전투기능 유지: 예비대는 지상군의 전투기능을 유지하여 지속적인 방어와 화력운용을 가능하게 한다
- 공격전환 기회 창출: 적의 주력부대를 제거하거나 교란으로 전환시켜 후속 작전계획 실행 기회를 제공한다
- 화력·동원 조정: 예비대는 지휘관의 판단과 상급부대의 동원계획에 따라 화력·동원 조정을 수행한다
3.
운용 절차  
- 작전목표 설정: 작전목표와 상급부대 명령에 따라 예비대 적용 목적과 범위를 설정한다
- 위치 및 보호확보: 적 공격로와 교차로 근처에서 교차로 보호병력을 배치해 위치 확보한다
- 동원 준비: 정보·화력·동원정보망을 통해 신속한 동원준비 및 검토절차를 수행한다
4.
지휘관 고려사항  
- 적 행동예측과 상급부대 계획검토 결과를 반영해 예비대로 활용할 부대와 적용시기를 결정해야 한다
- 예비대로 임무할 부대는 현재작전상황과 향후작전계획 검토결과에 따라 현행 임무 변경 또는 해제 여부를 확인해야 한다
5.
유의사항  
- 정찰정보 확인 없이 단독으로 공격하지 않으며 이동 중에는 정보·화력보호 조치가 필요하다
(일반 원칙 기준)

=== VALIDATION ===
{'has_think_tag': False, 'has_chat_template_token': False, 'has_required_sections': False, 'possible_fake_reference': False, 'repetition_ratio': 0.0, 'length': 619}


In [ ]:
navy_answer = ask(
    "navy",
    "너는 해군 교리 전문 AI 참모이다. 해상작전, 함대 운용, 해상통제 관점에서 답변하라.",
    "해상 경계작전 기본지침에 대해 알려줘.",
    max_tokens=700,
    return_raw=True,
)

print("=== CLEANED ANSWER ===")
print(navy_answer["answer"])

print("\n=== VALIDATION ===")
print(navy_answer["validation"])

=== CLEANED ANSWER ===
1.
개요  
해상 경계작전은 해양 영토 보호와 해양 이동 통제를 목표로 하는 작전으로서 주로 해군 함대와 항공력이 주력 부대다.
이는 단순한 방어 작전을 넘어 전략적 목표 달성에 기여하는 핵심 임무로 수행된다.
2.
핵심 원칙  
- **위험 관리**: 작전 계획 시 위험을 정량화하고 대응책을 마련한다.
- **협력 중심**: 육군·공군·해병과의 정보 공유 및 연합작전 체계를 유지한다.
- **법무 검토**: 국제법(UNCLOS 등)과 관련 지휘관의 판단을 근거로 실행한다.
3.
운용 절차  
- 준비 단계: 작전 목적·권한·위험 평가를 명령부에서 정리한다(예: 참모가 작성한 OPORD).
- 실행 단계: 감시/경계/검색/정찰 순서로 진행하며 적 신고 시 즉시 지휘관에게 보고한다(예: ESM 또는 SAR 활동).
- 종료 단계: 상황 변화나 지휘관 명령에 따라 조치를 중단하거나 전환한다.
4.
지휘관 고려사항  
- 정보 수집 우선순위를 설정하고 실시간 상황 판단에 반영해야 한다(예: AIS 데이터 vs 무선 통신 확인).
- 부대 능력을 극한까지 사용하지 않도록 하며 위험 대비 능력을 유지해야 한다(예: 함정 수리 여유 확보).
5.
유의사항  
- 민간선박이나 어업선과 충돌할 경우 법무 검토 후 조치해야 한다(예: UNCLOS 1982 제10조 참조).

=== VALIDATION ===
{'has_think_tag': False, 'has_chat_template_token': False, 'has_required_sections': False, 'possible_fake_reference': False, 'repetition_ratio': 0.0, 'length': 657}


In [ ]:
air_answer = ask(
    "air",
    "너는 해군 교리 전문 AI 참모이다. 해상작전, 함대 운용, 해상통제 관점에서 답변하라.",
    "해상 경계작전 기본지침에 대해 알려줘.",
    max_tokens=700,
    return_raw=True,
)

print("=== CLEANED ANSWER ===")
print(air_answer["answer"])

print("\n=== VALIDATION ===")
print(air_answer["validation"])

=== CLEANED ANSWER ===
1.
개요
해상 경계작전은 적의 해상 이동, 항행, 작전 활동을 감시하고 제한하는 작전으로, 전투기, 항공모함, 잠수함, 해병 부대 등 다양한 부대를 결합하여 수행한다.
2.
핵심 원칙
- 지휘관 의도 우선: 지휘관의 전투목표와 전략적 목표를 기반으로 작전을 계획한다.
- 정보 우위 확보: 정찰·감시·정밀타격 부대를 활용해 적의 동향과 행동을 파악한다.
- 연합·연합군 협력: 공군·육군·해병부대와 정보 및 작전 상호작용을 통해 효과를 극대화한다.
- 위험관리: 위험과 보상을 평가하여 안정적인 작전 실행을 보장한다.
3.
운용 절차
- 준비단계: 목표 설정, 정보 수집, 부대 배치 및 공군 지원 조정을 수행한다.
- 실행단계: 정찰부대로 확인 후 필요 시 공군 또는 해병 부대가 타격 또는 제한 조치를 실시한다.
- 평가단계: 결과를 검토하고 필요한 조정 또는 추가 조치를 결정한다.
4.
지휘관 고려사항
- 전투목표와 전략적 목표에 맞는 목적과 성공 기준을 명확히 설정해야 한다.
- 위험과 보상을 평가하며 안정적인 작전 실행 방안을 마련해야 한다.
- 관련부대와의 상호작용 범위와 책임 범위를 명확히 하여 협업 효율성을 높인다.
5.
유의사항
- 실제 적용 시에는 최신 교리 검토 및 현행 법령 확인이 필요하다.
- 교리 내용은 실제 운영 지침으로 사용되기 위해 현행화 점검이 필요하다.

=== VALIDATION ===
{'has_think_tag': False, 'has_chat_template_token': False, 'has_required_sections': False, 'possible_fake_reference': False, 'repetition_ratio': 0.0, 'length': 669}


## 10. 로컬 DoctrineRAG Backend `.env`

In [ ]:
print("backend/.env 예시")
print()
print("LLM_PROVIDER=vllm")
print(f"VLLM_BASE_URL={PUBLIC_BASE_URL}/v1")
print("VLLM_API_KEY=EMPTY")
print("ARMY_MODEL=army")
print("NAVY_MODEL=navy")
print("AIR_MODEL=air")


## 11. Backend 호출 예시 코드

In [ ]:
BACKEND_CLIENT_EXAMPLE = """
import os
import httpx

VLLM_BASE_URL = os.getenv("VLLM_BASE_URL")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "EMPTY")

MODEL_BY_BRANCH = {
    "army": os.getenv("ARMY_MODEL", "army"),
    "navy": os.getenv("NAVY_MODEL", "navy"),
    "air_force": os.getenv("AIR_MODEL", "air"),
}

SYSTEM_PROMPT_BY_BRANCH = {
    "army": "너는 육군 교리 전문 AI 참모이다.",
    "navy": "너는 해군 교리 전문 AI 참모이다.",
    "air_force": "너는 공군 교리 전문 AI 참모이다.",
}

async def call_branch_llm(branch: str, question: str, context: str = "") -> str:
    model = MODEL_BY_BRANCH[branch]
    system_prompt = SYSTEM_PROMPT_BY_BRANCH[branch]

    user_content = question
    if context:
        user_content = f"다음 교리 근거를 참고하여 답변하라.\\n\\n[교리 근거]\\n{context}\\n\\n[질문]\\n{question}"

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
        "temperature": 0.2,
        "max_tokens": 1024,
    }

    async with httpx.AsyncClient(timeout=180) as client:
        response = await client.post(
            f"{VLLM_BASE_URL}/chat/completions",
            headers={"Authorization": f"Bearer {VLLM_API_KEY}"},
            json=payload,
        )
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"]
"""
print(BACKEND_CLIENT_EXAMPLE)


## 12. Multi-Agent Supervisor 호출 예시

In [ ]:
joint_question = "합동 화력전에서 육군과 공군 협조 방식을 설명하라."

army_for_joint = ask(
    "army",
    "너는 육군 교리 전문 AI 참모이다. 지상작전과 화력지원 관점에서 답변하라.",
    joint_question,
)

air_for_joint = ask(
    "air",
    "너는 공군 교리 전문 AI 참모이다. 항공작전과 공중화력 관점에서 답변하라.",
    joint_question,
)

print("===== Army Agent Answer =====")
print(army_for_joint)
print()
print("===== Air Agent Answer =====")
print(air_for_joint)


## 13. 서버 종료

In [ ]:
# 필요할 때만 실행하세요.
try:
    vllm_process.terminate()
    print("vLLM process terminated.")
except Exception as e:
    print("vLLM terminate error:", e)

try:
    ngrok.kill()
    print("ngrok tunnels killed.")
except Exception as e:
    print("ngrok kill error:", e)


vLLM process terminated.
ngrok tunnels killed.


## Troubleshooting

### CUDA out of memory
아래 값을 낮춰보세요.

```txt
--max-model-len 2048
--gpu-memory-utilization 0.85
```

### LoRA 로드 실패
각 LoRA 폴더에 아래 파일이 있는지 확인하세요.

```txt
adapter_config.json
adapter_model.safetensors
```

그리고 LoRA가 Qwen3-8B 기준으로 학습된 adapter인지 확인하세요.

### 404 model not found
요청의 `model` 값은 아래 중 하나여야 합니다.

```txt
army
navy
air
```

### ngrok 접속 실패
ngrok token을 확인하고 런타임 재시작 후 다시 실행하세요.
